# Governance - access rules, RLS, CLS, grants

- **Default**: every member of the reader group (`students`) sees all assets and volumes.
- **`access_rules`**: personal overrides, one row per user. A personal row replaces the group default:
  `symbol` limits the visible asset (`NULL` = all), `can_see_volume` shows or hides volumes.
- **Users outside the reader group without a personal row see nothing.**
- **RLS**: `rls_asset_filter` on both facts. The fact stores only `asset_key`, so the filter resolves the symbol through `dim_asset`.
- **CLS**: `cls_mask_volume` returns `NULL` for volume columns when the user may not see volumes.
- **Grants**: the reader group gets gold business tables, not `access_rules`.

In [0]:
%run ../config/config

### 1. Access rules (personal overrides)


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ACCESS_RULES} (
    user_email     STRING  NOT NULL COMMENT 'Compared with current_user()',
    symbol         STRING           COMMENT 'Asset the user may see; NULL means all assets',
    can_see_volume BOOLEAN NOT NULL COMMENT 'TRUE if the user may see volume columns'
)
COMMENT 'Personal overrides for row filters and column masks on gold facts; replaces the reader-group default. Not granted to readers.'
""")

spark.sql(f"DELETE FROM {ACCESS_RULES} WHERE user_email = '{REVIEWER_EMAIL}'")
spark.sql(f"INSERT INTO {ACCESS_RULES} (user_email, symbol, can_see_volume) VALUES ('{REVIEWER_EMAIL}', NULL, TRUE)")

### 2. Row-level security
User with personal rows - only the assets in those rows. Otherwise - all assets if the user is in the reader group, nothing if not.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {RLS_FUNCTION}(fact_asset_key BIGINT)
RETURNS BOOLEAN
COMMENT 'Row filter: personal rule from access_rules if present, otherwise full access for the reader group'
RETURN
    -- personal rule for this asset
    EXISTS (
        SELECT 1
        FROM {ACCESS_RULES} r
        CROSS JOIN {GOLD}.dim_asset a
        WHERE a.asset_key = fact_asset_key
          AND r.user_email = current_user()
          AND (r.symbol IS NULL OR r.symbol = a.symbol)
    )
    -- no personal rule at all: reader-group default
    OR (
        NOT EXISTS (SELECT 1 FROM {ACCESS_RULES} r WHERE r.user_email = current_user())
        AND is_account_group_member('{READER_GROUP}')
    )
""")

for table in FILTERED_FACTS:
    spark.sql(f"ALTER TABLE {table} SET ROW FILTER {RLS_FUNCTION} ON (asset_key)")

### 3. Column-level security
User with personal rows - volumes only if one of them has `can_see_volume = TRUE`. Otherwise - volumes for the reader group, `NULL` for everyone else.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CLS_FUNCTION}(volume_value DOUBLE)
RETURNS DOUBLE
COMMENT 'Column mask: personal rule from access_rules if present, otherwise visible for the reader group'
RETURN CASE
    -- personal rule that allows volumes
    WHEN EXISTS (SELECT 1 FROM {ACCESS_RULES} r
                 WHERE r.user_email = current_user() AND r.can_see_volume)
    THEN volume_value
    -- no personal rule at all: reader-group default
    WHEN NOT EXISTS (SELECT 1 FROM {ACCESS_RULES} r WHERE r.user_email = current_user())
         AND is_account_group_member('{READER_GROUP}')
    THEN volume_value
    ELSE NULL
END
""")

for table, column in MASKED_COLUMNS:
    spark.sql(f"ALTER TABLE {table} ALTER COLUMN {column} SET MASK {CLS_FUNCTION}")

### 4. Grants

In [0]:
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {GOLD} TO `{READER_GROUP}`")
for table in READER_TABLES:
    spark.sql(f"GRANT SELECT ON TABLE {table} TO `{READER_GROUP}`")

### 5. Demonstration
The first cell adds a personal rule for the current user: only BTC, no volumes (it overrides the group default).
The second cell removes it, so the user falls back to the group default (everything visible).
Everyone else in the reader group keeps full access the whole time.

Do not run load or reconciliation notebooks while the rule is in place.

In [0]:
DEMO_SYMBOL = "BTC"

spark.sql(f"DELETE FROM {ACCESS_RULES} WHERE user_email = current_user()")
spark.sql(f"""
INSERT INTO {ACCESS_RULES} (user_email, symbol, can_see_volume)
VALUES (current_user(), '{DEMO_SYMBOL}', FALSE)
""")

In [0]:
spark.sql(f"DELETE FROM {ACCESS_RULES} WHERE user_email = current_user()")